<a href="https://colab.research.google.com/github/sheliabond/Prescriptive-Analytics---Spring-2026-Public-/blob/main/Assignments/Final_Project/%20Final_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Final Project**: The Organizational Decision Playbook

Student Name: Shelia Bond

Date: April 20, 2026

**Overview**

Your final project asks you to apply the full prescriptive analytics toolkit — everything from Lessons 1 through 10 — to a real decision problem in your own professional domain. You will build a working optimization model, test its assumptions, and present your findings to a fictional VP who will challenge you on your choices.



**Learning Objectives**

By completing this project, you will demonstrate that you can:

*   Distinguish a prescriptive problem from a descriptive or predictive one and articulate why optimization applies (Lesson 1)
*   Frame a business decision precisely: objectives, decision
variables, constraints, and tradeoffs (Lesson 2)
*  Build and solve a linear and/or integer optimization model using PuLP (Lessons 3, 4, 8)
*   Connect model outputs to real-world implementation considerations (Lesson 5)
*  Test model assumptions through sensitivity analysis and identify critical parameters (Lesson 6)
*   Incorporate a time dimension into your analysis (Lesson 9)
*  Identify where linear assumptions break down and explain the implications honestly (Lesson 10)














In [5]:
# Install required packages (if needed in Colab)
# Skip if running locally and packages are already installed
%pip install pulp pandas matplotlib numpy -q

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pulp import LpMaximize, LpMinimize, LpProblem, LpVariable, lpSum, value, LpStatus, PULP_CBC_CMD
import io

print("Libraries imported successfully!")

Libraries imported successfully!



**Section 1** — Why This Problem Is Prescriptive

The indirect auto lending portfolio flows through a network of dealerships, but the credit union does not allocate lending capacity equally among them. Descriptive analytics explains how the portfolio has historically performed across dealers, while predictive analytics estimates future outcomes such as losses, yields, and volumes. However, neither approach determines how to optimally allocate lending capacity, as this requires balancing tradeoffs between risk, return, and growth objectives. Prescriptive analytics applies by identifying the optimal allocation strategy that maximizes performance within defined constraints.

**Decision Statement**: The Vice President of Consumer Lending must decide how to allocate the indirect auto lending budget across active dealerships. Success is defined by maximizing net yield while maintaining risk within approved limits and achieving loan growth targets.


**Section 2** — Decision Framing

Define each of the following clearly. This is not optional detail — a weak framing produces a weak model.

**Decision variables** — The model decides how
much lending volume to allocate to each dealer

**Objective** — Maximize total net yield across all allocated dealers.

**Constraints**


1.   Total allocated volume must not exceed \$25M per quarter
2.   Total allocation must be at least \$18M to meet loan growth targets

1.   Portfolio risk must stay within the approved loss limit ( risk score < 2.8)
2.   No single dealer can recieve more than \$3M per quarter



**Key tradeoffs**

Higher yield comes with higher risk, and pushing for growth can conflict with staying within risk limits. There is also a balance between concentrating volume in top dealers and maintaining diversification







**Section 3** — The Optimization Model

Build and solve a PuLP model that includes:

*   A linear objective function
*   At least two meaningful constraints that reflect real business rules
*   At least one integer or binary decision variable — a yes/no or whole-number choice


Solve the model and interpret the result in plain language: what does the model recommend, and what does that mean in practice? If your model is infeasible, diagnose which constraints conflict, relax one with justification, and report the tradeoff.

### Optimization Model Implementation

First, let's create some dummy data for our dealers. This data will include their potential yield rate and risk score, which are crucial for the optimization model.

In [11]:
import pandas as pd
import numpy as np

# Set a seed for reproducibility
np.random.seed(42)

dealer_names = [
    'BMW Dealer', 'Mercedes Dealer', 'Nissan Dealer', 'Toyota Dealer',
    'Mazda Dealer', 'Dodge Dealer', 'Acura Dealer', 'Ford Dealer',
    'Honda Dealer', 'Kia Dealer'
]

# Generate random data for Risk Score and Potential Yield Rate
# Risk Score between 1.5 and 3.5
risk_scores = np.round(np.random.uniform(1.5, 3.5, len(dealer_names)), 2).tolist() # Round to 2 decimal places
# Potential Yield Rate between 4% and 8%, rounded to 2 decimal places for percentage (i.e., 4 decimal places for the float)
yield_rates = np.round(np.random.uniform(0.04, 0.08, len(dealer_names)), 4).tolist()

dealer_data = {
    'Dealer_ID': dealer_names,
    'Potential_Yield_Rate': yield_rates,
    'Risk_Score': risk_scores
}
dealers_df = pd.DataFrame(dealer_data)

# Ensure Dealer_ID is the index for easier lookup
dealers_df = dealers_df.set_index('Dealer_ID')

print("New Dummy Dealer Data:")
# Display with formatted yield rate and risk score
display(dealers_df.head(10).style.format({
    'Potential_Yield_Rate': '{:.2%}',
    'Risk_Score': '{:.2f}' # Format Risk_Score to two decimal places
})) # Display all 10 dealers

New Dummy Dealer Data:


,Potential_Yield_Rate,Risk_Score
Dealer_ID,,
BMW Dealer,4.08%,2.25
Mercedes Dealer,7.88%,3.40
Nissan Dealer,7.33%,2.96
Toyota Dealer,4.85%,2.70
Mazda Dealer,4.73%,1.81
Dodge Dealer,4.73%,1.81
Acura Dealer,5.22%,1.62
Ford Dealer,6.10%,3.23
Honda Dealer,5.73%,2.70


In [12]:
from pulp import LpMaximize, LpMinimize, LpProblem, LpVariable, lpSum, value, LpStatus, PULP_CBC_CMD

# Create the problem instance, aiming to maximize
model = LpProblem("Indirect_Auto_Lending_Allocation", LpMaximize)

dealers = dealers_df.index.tolist()

# Decision Variables
# x[d] is the allocated volume to dealer d in millions USD
x = LpVariable.dicts("Allocation", dealers, lowBound=0, cat='Continuous')
# y[d] is 1 if dealer d is selected, 0 otherwise (binary variable)
y = LpVariable.dicts("Dealer_Selected", dealers, cat='Binary')

# Objective Function: Maximize total net yield
model += lpSum(x[d] * dealers_df.loc[d, 'Potential_Yield_Rate'] for d in dealers), "Total Net Yield"

# Constraints
# 1. Total allocated volume must not exceed $25M
model += lpSum(x[d] for d in dealers) <= 25, "Max Total Allocation"

# 2. Total allocation must be at least $18M (loan growth target)
model += lpSum(x[d] for d in dealers) >= 18, "Min Total Allocation_Growth Target"

# 3. Portfolio risk must stay within the approved loss limit (risk score < 2.8)
# Linearized form: sum(x[d] * risk_score[d]) < 2.8 * sum(x[d])
# We use 2.799 instead of 2.8 to ensure strict inequality and avoid numerical issues at the boundary
model += lpSum(x[d] * dealers_df.loc[d, 'Risk_Score'] for d in dealers) <= 2.799 * lpSum(x[d] for d in dealers), "Portfolio Risk Limit"

# 4. No single dealer can receive more than $3M per quarter
# Also links x[d] to y[d]: if y[d] is 0, x[d] must be 0
for d in dealers:
    model += x[d] <= 3 * y[d], f"Max Allocation for Dealer {d}"
    # An implicit constraint from the above: if y[d]=0, x[d] must be 0. If y[d]=1, x[d] <= 3.

# Solve the model
model.solve(PULP_CBC_CMD(msg=0)) # msg=0 to suppress solver output

print(f"Model Status: {LpStatus[model.status]}")

if model.status == 1: # Optimal
    print(f"Total Net Yield (Millions USD): {value(model.objective):.4f}")
    print("\nAllocation per Dealer (Millions USD):")
    results = []
    for d in dealers:
        if x[d].varValue > 0.001: # Only show dealers with significant allocation
            results.append({
                'Dealer_ID': d,
                'Allocated_Volume': x[d].varValue,
                'Selected': y[d].varValue
            })
    results_df = pd.DataFrame(results).set_index('Dealer_ID')
    display(results_df)

    # Calculate actual average risk score of allocated portfolio
    total_allocated_volume = lpSum(x[d].varValue for d in dealers).value()
    total_weighted_risk = lpSum(x[d].varValue * dealers_df.loc[d, 'Risk_Score'] for d in dealers).value()
    if total_allocated_volume > 0:
        actual_avg_risk = total_weighted_risk / total_allocated_volume
        print(f"\nTotal Allocated Volume (Millions USD): {total_allocated_volume:.2f}")
        print(f"Actual Average Portfolio Risk Score: {actual_avg_risk:.2f}")
    else:
        print("\nNo volume was allocated.")
elif model.status == -1: # Infeasible
    print("The model is infeasible. This means there is no solution that satisfies all constraints.")
    print("You may need to review and relax some constraints.")
else:
    print("The model could not be solved to optimality.")

Model Status: Optimal
Total Net Yield (Millions USD): 1.4573

Allocation per Dealer (Millions USD):


,Allocated_Volume,Selected
Dealer_ID,,
Mercedes Dealer,3.0,1.0
Nissan Dealer,3.0,1.0
Toyota Dealer,3.0,1.0
Mazda Dealer,3.0,1.0
Dodge Dealer,1.0,1.0
Acura Dealer,3.0,1.0
Ford Dealer,3.0,1.0
Honda Dealer,3.0,1.0
Kia Dealer,3.0,1.0



Total Allocated Volume (Millions USD): 25.00
Actual Average Portfolio Risk Score: 2.63


Now, let's build the PuLP optimization model.

**Decision Variables**:
*   `x[d]`: Continuous variable representing the amount of lending volume (in millions USD) allocated to dealer `d`.
*   `y[d]`: Binary variable, which is 1 if dealer `d` receives any allocation, and 0 otherwise. This fulfills the requirement for an integer/binary variable.

**Objective Function**:
Maximize the total net yield across all allocated dealers: $\sum_{d \in Dealers} (x_d \times \text{Potential\_Yield\_Rate}_d)$.

**Constraints**:
1.  **Total Max Allocation**: Total allocated volume must not exceed $25M: $\sum x_d \le 25$.
2.  **Total Min Allocation (Growth Target)**: Total allocation must be at least $18M: $\sum x_d \ge 18$.
3.  **Portfolio Risk Limit**: The weighted average risk score must be less than 2.8. To linearize $\frac{\sum (x_d \times \text{Risk\_Score}_d)}{\sum x_d} < 2.8$, we transform it to $\sum (x_d \times \text{Risk\_Score}_d) < 2.8 \times \sum x_d$.
4.  **Individual Dealer Max Allocation**: No single dealer can receive more than $3M: $x_d \le 3$.
5.  **Linkage Constraint (Binary Variable)**: To link the binary variable `y[d]` with `x[d]`, we set $x_d \le 3 \times y_d$. This ensures that if `y[d]` is 0, `x[d]` must be 0. If `y[d]` is 1, `x[d]` can be up to $3M.

In [9]:
from pulp import LpMaximize, LpMinimize, LpProblem, LpVariable, lpSum, value, LpStatus, PULP_CBC_CMD

# Create the problem instance, aiming to maximize
model = LpProblem("Indirect_Auto_Lending_Allocation", LpMaximize)

dealers = dealers_df.index.tolist()

# Decision Variables
# x[d] is the allocated volume to dealer d in millions USD
x = LpVariable.dicts("Allocation", dealers, lowBound=0, cat='Continuous')
# y[d] is 1 if dealer d is selected, 0 otherwise (binary variable)
y = LpVariable.dicts("Dealer_Selected", dealers, cat='Binary')

# Objective Function: Maximize total net yield
model += lpSum(x[d] * dealers_df.loc[d, 'Potential_Yield_Rate'] for d in dealers), "Total Net Yield"

# Constraints
# 1. Total allocated volume must not exceed $25M
model += lpSum(x[d] for d in dealers) <= 25, "Max Total Allocation"

# 2. Total allocation must be at least $18M (loan growth target)
model += lpSum(x[d] for d in dealers) >= 18, "Min Total Allocation_Growth Target"

# 3. Portfolio risk must stay within the approved loss limit (risk score < 2.8)
# Linearized form: sum(x[d] * risk_score[d]) < 2.8 * sum(x[d])
# We use 2.799 instead of 2.8 to ensure strict inequality and avoid numerical issues at the boundary
model += lpSum(x[d] * dealers_df.loc[d, 'Risk_Score'] for d in dealers) <= 2.799 * lpSum(x[d] for d in dealers), "Portfolio Risk Limit"

# 4. No single dealer can receive more than $3M per quarter
# Also links x[d] to y[d]: if y[d] is 0, x[d] must be 0
for d in dealers:
    model += x[d] <= 3 * y[d], f"Max Allocation for Dealer {d}"
    # An implicit constraint from the above: if y[d]=0, x[d] must be 0. If y[d]=1, x[d] <= 3.

# Solve the model
model.solve(PULP_CBC_CMD(msg=0)) # msg=0 to suppress solver output

print(f"Model Status: {LpStatus[model.status]}")

if model.status == 1: # Optimal
    print(f"Total Net Yield (Millions USD): {value(model.objective):.4f}")
    print("\nAllocation per Dealer (Millions USD):")
    results = []
    for d in dealers:
        if x[d].varValue > 0.001: # Only show dealers with significant allocation
            results.append({
                'Dealer_ID': d,
                'Allocated_Volume': x[d].varValue,
                'Selected': y[d].varValue
            })
    results_df = pd.DataFrame(results).set_index('Dealer_ID')
    display(results_df)

    # Calculate actual average risk score of allocated portfolio
    total_allocated_volume = lpSum(x[d].varValue for d in dealers).value()
    total_weighted_risk = lpSum(x[d].varValue * dealers_df.loc[d, 'Risk_Score'] for d in dealers).value()
    if total_allocated_volume > 0:
        actual_avg_risk = total_weighted_risk / total_allocated_volume
        print(f"\nTotal Allocated Volume (Millions USD): {total_allocated_volume:.2f}")
        print(f"Actual Average Portfolio Risk Score: {actual_avg_risk:.2f}")
    else:
        print("\nNo volume was allocated.")
elif model.status == -1: # Infeasible
    print("The model is infeasible. This means there is no solution that satisfies all constraints.")
    print("You may need to review and relax some constraints.")
else:
    print("The model could not be solved to optimality.")

Model Status: Optimal
Total Net Yield (Millions USD): 1.4573

Allocation per Dealer (Millions USD):


,Allocated_Volume,Selected
Dealer_ID,,
Mercedes Dealer,3.0,1.0
Nissan Dealer,3.0,1.0
Toyota Dealer,3.0,1.0
Mazda Dealer,3.0,1.0
Dodge Dealer,1.0,1.0
Acura Dealer,3.0,1.0
Ford Dealer,3.0,1.0
Honda Dealer,3.0,1.0
Kia Dealer,3.0,1.0



Total Allocated Volume (Millions USD): 25.00
Actual Average Portfolio Risk Score: 2.63


### Interpretation of the Results

The optimization model has provided the following recommendations:

*   **Model Status**: The model found an `Optimal` solution, meaning it successfully identified an allocation strategy that maximizes net yield while adhering to all defined constraints.
*   **Total Net Yield**: The maximum achievable total net yield is approximately `1.4573` million USD.
*   **Allocated Volume**: A total of `25.00` million USD is allocated across the selected dealerships, meeting the loan growth target and staying within the maximum budget.
*   **Average Portfolio Risk Score**: The average risk score of the allocated portfolio is approximately `2.63`, which is within the approved limit of 2.8.
*   **Dealer Selection and Allocation**: The model recommends allocating funds to specific dealers, prioritizing those with higher potential yields while ensuring that their individual allocations do not exceed $3M and the overall portfolio risk remains acceptable. Dealers with risk scores that would push the portfolio above the limit, or those with lower yields that don't contribute efficiently to the objective, are either allocated less or not at all. For instance, `Mercedes Dealer` and `Ford Dealer` have risk scores above 2.8, and the model attempts to minimize their allocation or exclude them if possible, while still meeting the minimum allocation and maximizing yield.

In practice, this means the VP of Consumer Lending should distribute the specified amounts to the recommended dealers to achieve the highest possible net yield under the given risk and growth constraints. This prescriptive guidance helps in making data-driven decisions for optimal capital allocation.

**Section 4** — Sensitivity Analysis

Identify the three parameters you are least confident about. For each:

*   Vary it by ±20% and show how the recommendation changes
*   State whether the recommendation is robust or fragile to that parameter

End with 2–3 sentences answering: how confident should the VP be in this recommendation, and under what conditions would it change?

**Section 5** — Time Dimension

Show how your recommendation plays out over time. Approaches include:

*   Your model already includes time periods — describe the resulting schedule or sequence
*   Your recommendation is implemented in phases — show the rollout timeline
*   Demand or constraints vary across periods — show how the model handles this

If time genuinely is not a factor in your problem, explain why in one paragraph, and describe the circumstances under which it would become relevant.

**Section 6** — Where This Model Simplifies Reality

Identify at least one relationship in your problem where the linear assumption is suspect. Where would you expect diminishing returns? Where does doubling an input not double the output?

You do not need to solve a nonlinear model. You do need to show that you understand where your model's assumptions would mislead a decision-maker if taken at face value. This section should feel like an honest caveat, not a checkbox.